# Muse Dataset (To find emotions of songs)

In [ ]:
import pandas
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Read the data 
muse = pandas.read_csv('muse_v3.csv')
muse.head()

In [ ]:
# Drop columns with missing values
muse = muse.dropna(axis=1)

# Drop columns except track, artist and seeds
columns_to_keep = ['track', 'artist', 'seeds']
muse = muse[columns_to_keep]
muse

In [ ]:
# Find rows with only multiple seeds value and drop them
print(f"Before dropping rows with multiple seed values {len(muse['seeds'])}")

multiple_seed_rows = muse[muse['seeds'].str.contains(',')]
print(f"Rows with multiple seed values {len(multiple_seed_rows)}")

muse = muse.drop(multiple_seed_rows.index)
print(f"After dropping rows with multiple seed values {len(muse['seeds'])}")

muse.head()

In [ ]:
# Finding the top 10 seeds
top_seeds = muse.groupby('seeds').size().sort_values(ascending=False).head(10)
print(top_seeds)

In [ ]:
# Finding how many rows have the top 10 seeds
rows_with_top_seeds = top_seeds.index
rows_with_top_seeds = muse[muse['seeds'].isin(rows_with_top_seeds)]
print(f"Rows with top 10 seeds: {len(rows_with_top_seeds)}")

In [ ]:
# Plotting the top 10 seeds in rows_with_top_seeds
sns.countplot(y='seeds', data=rows_with_top_seeds)
plt.show()

In [ ]:
print(len(rows_with_top_seeds))
print(len(muse))

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
rows_with_top_seeds = rows_with_top_seeds.copy()
rows_with_top_seeds.loc[:, 'seeds'] = label_encoder.fit_transform(rows_with_top_seeds['seeds'])
rows_with_top_seeds.head()


In [ ]:
# Show how many types of seeds are there
print(f"Number of types of seeds: {len(rows_with_top_seeds['seeds'].unique())}")

In [ ]:
# One hot encoding the seeds
#one_hot_seeds = pandas.get_dummies(rows_with_top_seeds['seeds'])
#one_hot_seeds.head()

In [ ]:
# print the row where the track is Whip My Hair
#muse[muse['track'] == 'Whip My Hair']

In [ ]:
# Applying label encoding to the seeds
# rows_with_top_seeds = rows_with_top_seeds.drop('seeds', axis=1)
# rows_with_top_seeds = rows_with_top_seeds.join(one_hot_seeds)
# rows_with_top_seeds.head()


In [ ]:
# Checking if we applied one hot encoding correctly
#display(rows_with_top_seeds[rows_with_top_seeds['track'] == 'Whip My Hair'])
#display(muse[muse['track'] == 'Whip My Hair'])

# Genius API (To get lyrics of songs)

In [ ]:
# https://lyricsgenius.readthedocs.io/en/master/index.html

!pip install git+https://github.com/johnwmillr/LyricsGenius.git

In [ ]:
# Your Genius access token
token = input("Enter your Genius access token: ")

In [ ]:
from lyricsgenius import Genius
genius = Genius(token)

In [ ]:
# Turn off status messages
genius.verbose = False

# Remove section headers (e.g. [Chorus]) from lyrics when searching
genius.remove_section_headers = True

# Include hits thought to be non-songs (e.g. track lists)
genius.skip_non_songs = False

# Exclude songs with these words in their title
genius.excluded_terms = ["(Remix)", "(Live)"]

In [ ]:
# get the first row of the rows_with_top_seeds dataframe
#rows_with_top_seeds.iloc[0]['artist']

In [ ]:
# drop all lyrics columns
#rows_with_top_seeds = rows_with_top_seeds.drop(['lyrics'], axis=1)

In [ ]:
# Defining a function to cleaning the punctuation from the lyrics
from string import punctuation

def clean_lyrics(lyrics):
    lyrics = [line.lower() for line in lyrics]
    lyrics = [' '.join(line.split()) for line in lyrics]
    lyrics = [' '.join([word for word in line.split() if word.strip()]) for line in lyrics]
    lyrics = [' '.join([word for word in line.split() if word.strip() and word not in punctuation]) for line in lyrics]
    # Removing non-ascii characters
    lyrics = [line.encode('ascii', 'ignore').decode('ascii') for line in lyrics]
    lyrics = ' '.join(lyrics)
    
    return lyrics

In [ ]:
# Defining a function that returns the cleaned lyrics of a song
def get_lyrics(artist, track):
    song = genius.search_song(track, artist,)
    if song is None:
        print(f"Could not find lyrics for {track} by {artist}")
        return None
    lines = song.lyrics.split('\n')
    # Skip the first line because it is irrelevant
    lyrics = '\n'.join(lines[1:])
    lyrics = lyrics.split()

    lyrics = clean_lyrics(lyrics)
    return lyrics

In [ ]:
# An example of how to use the get_lyrics function
get_lyrics(rows_with_top_seeds.iloc[0]['artist'], rows_with_top_seeds.iloc[0]['track'])

In [ ]:
# Shuffle the rows_with_top_seeds dataframe
rows_with_top_seeds = rows_with_top_seeds.sample(frac=1).reset_index(drop=True)
rows_with_top_seeds.head()

In [ ]:
# Iterate over rows with iterrows()
# and add the lyrics for each row

counter = 0
# Iterate 300 rows
for index, row in rows_with_top_seeds.iterrows():
    if counter >= 300:
        break
    print(f'Iterating over row {index} out of {len(rows_with_top_seeds)}')
    artist = row['artist']
    track = row['track']
    counter += 1
    # if get_lyrics raises an error, skip the row
    try:
        lyrics = get_lyrics(artist, track)
        if lyrics is not None:
            rows_with_top_seeds.at[index, 'lyrics'] = lyrics
        else:
            print(f"Could not find lyrics for {track} by {artist}")
            rows_with_top_seeds.at[index, 'lyrics'] = None
    except:
        print(f"Could not find lyrics for {track} by {artist}")
        rows_with_top_seeds.at[index, 'lyrics'] = None

    

In [ ]:
# drop lyrics column
#rows_with_top_seeds = rows_with_top_seeds.drop(['lyrics'], axis=1)

In [ ]:
rows_with_top_seeds

In [ ]:
# Drop columns with Nan values for lyrics
rows_with_top_seeds = rows_with_top_seeds.dropna(subset=['lyrics'])
rows_with_top_seeds

In [ ]:
# Drop track and artist columns
rows_with_top_seeds = rows_with_top_seeds.drop(['track', 'artist'], axis=1)
rows_with_top_seeds

In [ ]:
# Save the dataframe to a csv file
rows_with_top_seeds.to_csv('final_dataset.csv', index=False)